# Конспект: функции и ООП на примере шахмат

Краткая систематизация основных идей: функции, классы, наследование, полиморфизм и работа объектов друг с другом.


## 1. Проверка координат

Обычная функция проверяет, находится ли клетка внутри шахматной доски.


**Код:** условие объединяет две проверки диапазона — отдельно для строки и столбца.


In [ ]:
def is_inside(row: int, col: int) -> bool:
    """Проверить, находится ли клетка (row, col) внутри шахматной доски."""
    return 0 <= row <= 7 and 0 <= col <= 7


## 2. Правило хода ладьи без ООП

До появления классов координаты фигуры передаются в функцию явно.


**Код:** сначала проверяются границы доски, затем правило движения ладьи по одной строке или одному столбцу.


In [ ]:
def rook_can_move(
    row: int,
    col: int,
    new_row: int,
    new_col: int
    ) -> bool:
    """
    Проверить, может ли ладья перейти из клетки (row, col)
    в клетку (new_row, new_col).

    Метод должен учитывать:
    - границы доски;
    - запрет хода в ту же клетку;
    - движение только по строке или столбцу.
    """
    can_move_1 = is_inside(row, col) and is_inside(new_row, new_col)
    can_move_2 = ((row == new_row and col != new_col) or (row != new_row and col == new_col))
    return can_move_1 and can_move_2



## 3. Базовый класс `Piece`

**Класс** описывает данные и поведение объектов. **Объект** — конкретный экземпляр класса. `__init__` задаёт начальное состояние, а `self` указывает на текущий объект.


**Код:** объект хранит цвет и координаты; методы позволяют узнать и изменить позицию.


In [ ]:
class Piece:
    def __init__(self, color: str, row: int, col: int):
        """Сохранить цвет фигуры и её начальную позицию."""
        self.color = color
        self.row = row
        self.col = col

    def position(self) -> tuple[int, int]:
        """Вернуть текущие координаты фигуры."""
        return self.row, self.col

    def move_to(self, row: int, col: int) -> None:
        """Изменить позицию фигуры на указанную клетку. Без проверки на возможность движения."""
        self.row = row
        self.col = col


## 4. Специальный метод `__str__`

`__str__` определяет, как объект представляется при `print(obj)`. В базовом классе также появляется общий интерфейс `can_move()` и `char()`, который затем реализуют дочерние классы.


**Код:** `move_to()` вызывает `can_move()` перед изменением координат, а `__str__()` формирует строковое представление объекта.


In [ ]:
class Piece:
    def __init__(self, color: str, row: int, col: int):
        """Сохранить цвет фигуры и её начальную позицию."""
        self.color = color
        self.row = row
        self.col = col

    def can_move(self, new_row: int, new_col: int) -> bool:
        pass

    def position(self) -> tuple[int, int]:
        """Вернуть текущие координаты фигуры."""
        return self.row, self.col

    def move_to(self, row: int, col: int) -> bool:
        """Изменить позицию фигуры на указанную клетку. Без проверки на возможность движения."""
        if not self.can_move(row, col):
            return False
        self.row = row
        self.col = col
        return True

    def __str__(self) -> str:
        """Вернуть строковое обозначение фигуры для вывода на доске."""
        return f"Фигура {self.__class__.__name__} {self.color} цвета на позиции {self.row}, {self.col}."

    def char(self):
        pass

p = Piece("WHITE", 1, 7)
print(p)

## 5. Наследование: ладья

`Rook` наследует общие поля и методы `Piece`, но задаёт собственные правила хода и обозначение фигуры.


**Код:** метод `can_move()` использует координаты объекта через `self`; `char()` возвращает обозначение ладьи.


In [ ]:
class Rook(Piece):
    def can_move(self,
        new_row: int,
        new_col: int
        ) -> bool:
        """
        Проверить, может ли ладья перейти из клетки (row, col)
        в клетку (new_row, new_col).
    
        Метод должен учитывать:
        - границы доски;
        - запрет хода в ту же клетку;
        - движение только по строке или столбцу.
        """
        can_move_1 = is_inside(self.row, self.col) and is_inside(new_row, new_col)
        can_move_2 = ((self.row == new_row and self.col != new_col) or (self.row != new_row and self.col == new_col))
        return can_move_1 and can_move_2

    def char(self):
        return "R"

r1 = Rook("WHITE", 1, 1)
print(r1.move_to(5, 9))
print(r1)

## 6. Ещё один наследник: конь

Разные классы могут иметь одинаковый набор методов, но разную реализацию.


**Код:** ход коня проверяется по изменениям координат `2×1` или `1×2`, после чего дополнительно проверяются границы доски.


In [ ]:
class Knight(Piece):
    def can_move(self, row: int, col: int) -> bool:
        """
        Проверить возможность хода коня.

        Конь должен менять одну координату на 1,
        а другую — на 2.
        """
        can_move_1 = (abs(self.row - row) == 2 and abs(self.col - col) == 1) or (abs(self.row - row) == 1 and abs(self.col - col) == 2)
        can_move_2 = is_inside(self.row, self.col) and is_inside(row, col)
        return can_move_1 and can_move_2

    def char(self):
        return "K"
        



## 7. Полиморфизм

Для разных фигур можно одинаково вызывать `position()`, `can_move()` и `move_to()`. Конкретная реализация метода определяется классом объекта.


## 8. Функция как объект

Функцию можно хранить в переменной, передавать как аргумент и использовать как условие отбора.


**Код:** функция-предикат возвращает `True` только для белых фигур.


In [ ]:
def is_white(piece):
    """Вернуть True, если фигура белая."""
    return piece.color == "WHITE"


## 9. Класс шахматной доски

`Board` хранит состояние доски и предоставляет операции для добавления, получения и отображения фигур.


**Код:** двумерный список моделирует клетки доски; при выводе используется метод `char()` конкретной фигуры.


In [ ]:
class Board:
    def __init__(self):
        """Создать пустую шахматную доску и хранилище фигур."""
        self.table = [[None] * 8 for i in range(8)]

    def add_piece(self, piece):
        """Добавить фигуру на доску."""
        self.table[piece.row][piece.col] = piece

    def get_piece(self, row: int, col: int):
        """Вернуть фигуру на указанной клетке или None, если клетка пуста."""
        return self.table[row][col]

    def is_empty(self, row: int, col: int) -> bool:
        """Проверить, свободна ли указанная клетка."""
        return self.table[row][col] != None

    def show(self) -> None:
        """
        Вывести доску 8×8.

        Для пустой клетки вывести точку,
        для занятой — строковое обозначение фигуры.
        """
        print("", end="\t")
        for col in range(8):
            print(col, end="\t")
        print()
        print()
        print()
        for row in range(8):
            print(row, end="\t")
            for col in range(8):
                if self.table[row][col] == None:
                    print(".", end="\t")
                else:
                    if self.table[row][col].color == "WHITE":
                        print("w" + self.table[row][col].char(), end="\t")
                    elif self.table[row][col].color == "BLACK":
                        print("b" + self.table[row][col].char(), end="\t")
            print()
            print()
            print()


board = Board()
board.add_piece(Knight("WHITE", 1, 7))
board.add_piece(Rook("BLACK", 2, 3))
board.show()

## 10. Обычная функция, работающая с объектом

Не каждое действие нужно превращать в метод класса. Функция `possible_moves()` перебирает клетки и использует общий интерфейс `can_move()`.


**Код:** для каждой клетки проверяется возможность хода; допустимые координаты добавляются в список.


In [ ]:
def possible_moves(piece) -> list[tuple[int, int]]:
    """
    Найти все клетки доски, в которые фигура
    геометрически может сделать ход.

    Состояние других фигур на доске пока не учитывать.
    """
    lst = []
    for row in range(8):
        for col in range(8):
            if piece.can_move(row, col):
                lst.append((row, col))
    return lst

## 11. Слон

Слон ходит по диагонали: абсолютные изменения строки и столбца должны совпадать.


**Код:** дополнительно проверяются границы доски и запрет хода в ту же клетку.


In [ ]:
class Bishop(Piece):
    def can_move(self, row: int, col: int) -> bool:
        can_move_1 = abs(row - self.row) == abs(col - self.col)
        can_move_2 = is_inside(self.row, self.col) and is_inside(row, col)
        can_move_3 = not(row == self.row and col == self.col)
        return can_move_1 and can_move_2 and can_move_3

    def char(self):
        return "B"


**Проверка:** несколько вызовов показывают допустимые и недопустимый ходы слона.


In [ ]:
bishop = Bishop("WHITE", 2, 2)

# Должно быть:
# True
# True
# False

print(bishop.can_move(5, 5))
print(bishop.can_move(0, 4))
print(bishop.can_move(2, 5))


## 12. Ферзь

Ферзь объединяет правила ладьи и слона.


**Код:** ход разрешён, если выполняется правило ладьи **или** правило слона, а конечная клетка находится на доске.


In [ ]:
class Queen(Piece):
    def can_move(self, row: int, col: int) -> bool:
        can_move_bishop_1 = abs(row - self.row) == abs(col - self.col)
        can_move_bishop_2 = not(row == self.row and col == self.col)
        can_move_rook_1 = ((self.row == row and self.col != col) or (self.row != row and self.col == col))
        can_move_1 = is_inside(self.row, self.col) and is_inside(row, col)
        return can_move_1 and (can_move_rook_1 or can_move_bishop_1 and can_move_bishop_2)

    def char(self):
        return "Q"


## 13. Пешка

Поведение объекта может зависеть от его состояния: направление движения пешки определяется цветом.


**Код:** белая и чёрная пешки движутся в противоположных направлениях; с начальной строки разрешён ход на одну или две клетки.


In [ ]:
class Pawn(Piece):
    def can_move(self, row: int, col: int) -> bool:
        if not(is_inside(self.row, self.col) and is_inside(row, col)):
            return False
            
        if self.color == "WHITE":
            if self.row == 1:
                return ((self.row + 1 == row) or (self.row + 2 == row)) and (self.col == col)
            else:
                return (self.row + 1 == row) and (self.col == col)
        elif self.color == "BLACK":
            if self.row == 6:
                return ((self.row - 1 == row) or (self.row - 2 == row)) and (self.col == col)
            else:
                return (self.row - 1 == row) and (self.col == col)

    def char(self):
        return "P"


# Что ещё нужно реализовать для полноценной симуляции шахмат

Сейчас программа умеет описывать отдельные фигуры и проверять их **геометрические** ходы. Этого достаточно для изучения базовых идей ООП, но ещё недостаточно для настоящей партии.

Ниже оставлены заготовки, которые нужно будет дописать на семинаре. Лучше реализовывать их именно в этом порядке.


## 14. Король

В наборе фигур пока нет короля. Он ходит на одну клетку по горизонтали, вертикали или диагонали.

Пока здесь проверяется только форма хода. Проверку шаха добавим отдельно.


In [ ]:
class King(Piece):
    def can_move(self, row: int, col: int) -> bool:
        # TODO: проверить границы доски
        # TODO: запретить ход в ту же клетку
        # TODO: разрешить смещение максимум на 1 по каждой координате
        pass

    def char(self):
        return "K"


## 15. Доска должна участвовать в проверке хода

Сейчас `piece.can_move()` ничего не знает о других фигурах. Поэтому ладья, слон или ферзь могут «проходить» сквозь фигуры.

Нужно научить доску определять, свободен ли путь между начальной и конечной клетками.


In [ ]:
class Board:
    # Остальные методы класса остаются такими же.

    def is_path_clear(
        self,
        start_row: int,
        start_col: int,
        end_row: int,
        end_col: int
    ) -> bool:
        # TODO:
        # 1. определить направление движения по строке;
        # 2. определить направление движения по столбцу;
        # 3. пройти по всем промежуточным клеткам;
        # 4. вернуть False, если встретилась фигура;
        # 5. иначе вернуть True.
        pass


## 16. Нельзя вставать на клетку со своей фигурой

Даже если геометрически ход допустим, конечная клетка может быть занята фигурой того же цвета.

Эту проверку удобнее выполнять на уровне доски, а не внутри каждого класса фигуры.


In [ ]:
class Board:
    # Остальные методы класса остаются такими же.

    def can_land_on(self, piece: Piece, row: int, col: int) -> bool:
        # TODO:
        # - если клетка пустая, ход на неё разрешён;
        # - если там фигура противника, её можно взять;
        # - если там своя фигура, ход запрещён.
        pass


## 17. Единый метод выполнения хода

Сейчас `Piece.move_to()` меняет только координаты объекта. Но в настоящей игре одновременно нужно изменить и состояние `Board`.

Поэтому нужен единый метод доски: он проверяет ход, убирает взятую фигуру и переносит текущую.


In [ ]:
class Board:
    # Остальные методы класса остаются такими же.

    def move_piece(
        self,
        start_row: int,
        start_col: int,
        end_row: int,
        end_col: int
    ) -> bool:
        # TODO:
        # 1. найти фигуру в начальной клетке;
        # 2. проверить существование фигуры;
        # 3. проверить геометрию хода;
        # 4. проверить конечную клетку;
        # 5. для ладьи, слона и ферзя проверить свободный путь;
        # 6. обработать взятие;
        # 7. изменить таблицу board.table;
        # 8. изменить координаты самой фигуры.
        pass


## 18. Полные правила пешки

Текущий `Pawn.can_move()` описывает обычное движение вперёд, но у пешки есть особые правила:

- вперёд она ходит только на свободную клетку;
- берёт фигуру по диагонали;
- двойной ход возможен только с начальной позиции и при свободной промежуточной клетке;
- существует взятие на проходе;
- на последней горизонтали пешка превращается в другую фигуру.

Поэтому часть логики пешки должна учитывать состояние доски.


In [ ]:
class Board:
    def can_pawn_move(
        self,
        pawn: Pawn,
        row: int,
        col: int
    ) -> bool:
        # TODO: обычный ход на одну клетку
        # TODO: первый ход на две клетки
        # TODO: проверить промежуточную клетку
        # TODO: взятие по диагонали
        # TODO: позже добавить взятие на проходе
        pass

    def promote_pawn(self, pawn: Pawn, new_piece_type):
        # TODO:
        # заменить пешку, дошедшую до последней горизонтали,
        # на Queen / Rook / Bishop / Knight.
        pass


## 19. Рокировка

Для рокировки недостаточно обычного правила движения короля.

Нужно помнить, двигались ли раньше король и соответствующая ладья, проверять свободные клетки между ними и убеждаться, что король не проходит через атакованную клетку.


In [ ]:
class Piece:
    def __init__(self, color: str, row: int, col: int):
        self.color = color
        self.row = row
        self.col = col

        # TODO: обновлять после первого выполненного хода
        self.has_moved = False


class Board:
    def can_castle(self, king: King, rook: Rook) -> bool:
        # TODO:
        # - проверить has_moved у короля и ладьи;
        # - проверить свободные клетки между ними;
        # - проверить, что король не находится под шахом;
        # - проверить клетки, через которые проходит король.
        pass


## 20. Атакованные клетки и шах

Чтобы запрещать королю ход под удар и реализовать рокировку, нужно уметь определять, атакует ли противник некоторую клетку.

После этого можно проверить, находится ли король под шахом.


In [ ]:
class Board:
    def is_square_attacked(
        self,
        row: int,
        col: int,
        by_color: str
    ) -> bool:
        # TODO:
        # перебрать фигуры указанного цвета
        # и определить, атакует ли хотя бы одна из них клетку.
        pass

    def find_king(self, color: str):
        # TODO: найти короля указанного цвета на доске.
        pass

    def is_check(self, color: str) -> bool:
        # TODO:
        # 1. найти короля;
        # 2. определить цвет соперника;
        # 3. проверить, атакуется ли клетка короля.
        pass


## 21. Нельзя делать ход, оставляющий своего короля под шахом

Даже геометрически правильный ход нельзя выполнять, если после него собственный король оказывается под ударом.

Один из простых вариантов реализации — временно выполнить ход, проверить положение короля, а затем при необходимости отменить ход.


In [ ]:
class Board:
    def is_legal_move(
        self,
        start_row: int,
        start_col: int,
        end_row: int,
        end_col: int
    ) -> bool:
        # TODO:
        # 1. выполнить все обычные проверки;
        # 2. временно применить ход;
        # 3. проверить шах своему королю;
        # 4. вернуть доску в исходное состояние;
        # 5. вернуть результат.
        pass


## 22. Мат и пат

После появления полной проверки допустимого хода можно перебрать все фигуры игрока и проверить, существует ли хотя бы один разрешённый ход.

- **Мат**: король под шахом и допустимых ходов нет.
- **Пат**: шаха нет, но допустимых ходов тоже нет.


In [ ]:
class Board:
    def legal_moves(self, color: str):
        # TODO: вернуть все допустимые ходы всех фигур данного цвета.
        pass

    def is_checkmate(self, color: str) -> bool:
        # TODO: шах + отсутствие допустимых ходов.
        pass

    def is_stalemate(self, color: str) -> bool:
        # TODO: нет шаха + отсутствие допустимых ходов.
        pass


## 23. Начальная расстановка фигур

Чтобы начинать настоящую партию, доска должна уметь сама создавать стандартную начальную позицию.


In [ ]:
class Board:
    def setup(self) -> None:
        # TODO:
        # - поставить белые пешки;
        # - поставить чёрные пешки;
        # - расставить ладьи, коней, слонов;
        # - поставить ферзей;
        # - поставить королей.
        pass


## 24. Класс `Game`

Последний уровень — объект, отвечающий уже не за отдельную фигуру и не только за доску, а за всю партию.

Он должен хранить очередь хода, историю партии и состояние игры.


In [ ]:
class Game:
    def __init__(self):
        self.board = Board()
        self.board.setup()

        self.current_color = "WHITE"
        self.history = []
        self.finished = False

    def move(
        self,
        start_row: int,
        start_col: int,
        end_row: int,
        end_col: int
    ) -> bool:
        # TODO:
        # 1. проверить, что ходит фигура current_color;
        # 2. проверить допустимость хода;
        # 3. выполнить ход;
        # 4. сохранить его в history;
        # 5. проверить мат / пат;
        # 6. передать ход другому игроку.
        pass

    def change_turn(self):
        # TODO: поменять WHITE <-> BLACK.
        pass


## 25. Что можно добавить после основной версии

После семинара базовую модель можно расширять дальше:

- взятие на проходе;
- выбор фигуры при превращении пешки;
- правило трёх повторений позиции;
- правило 50 ходов;
- недостаточный материал для мата;
- запись ходов;
- сохранение и загрузка партии;
- графический интерфейс.

Для основного семинара достаточно дойти до класса `Game`, проверки шаха и определения окончания партии.


## Главное

- Функции удобны для самостоятельных операций.
- Класс объединяет данные и связанные с ними действия.
- Наследование выносит общее поведение в базовый класс.
- Полиморфизм позволяет одинаково работать с объектами разных классов.
- Обычные функции и ООП можно свободно сочетать в одной программе.
